# Adidas LAM Chatbot Analytics

This notebook reproduces the core business-case analysis from the supplied Excel workbook. The workbook supports volume, channel, category, subcategory, classification-failure, and prioritization analysis. The containment, repeat-contact, and resolution KPIs are treated as business-case inputs because the workbook does not contain session IDs, customer IDs, timestamps, CSAT, or resolution outcome labels.

# Section 1 — Imports

In [66]:
from pathlib import Path
import pandas as pd
import numpy as np



# Section 2 — Configuration

In [67]:
DATA_PATH = Path(r"C:\Users\restr\Desktop\adidas-chatbot-case\data\raw\Business_Case_Chatbot_data_Raw_Data.xlsx")

if DATA_PATH.exists():
    print(f" Success! File found at: {DATA_PATH}")
else:
    print(f" ERROR: File NOT found at: {DATA_PATH}")
    print("Please double-check the folder path or spelling.")

 Success! File found at: C:\Users\restr\Desktop\adidas-chatbot-case\data\raw\Business_Case_Chatbot_data_Raw_Data.xlsx


# Section 3 — Cargar Dataset

In [68]:
# Abrir el archivo de Excel para inspeccionar la metadata
xls = pd.ExcelFile(DATA_PATH)

# Mostrar los nombres de las hojas del archivo
print("   Hojas disponibles en el archivo de excel:")
for index, name in enumerate(xls.sheet_names, start=1):
    print(f"  {index}. {name}")

   Hojas disponibles en el archivo de excel:
  1. Agent handled only volume
  2. Hybrid Handled only volume
  3. Bot only volume


# Section 4 — Cargar cada hoja

1. Agent handled only volume --> df_agent
2. Hybrid Handled only volume --> df_hybrid
3. Bot only volume --> df_bot

In [69]:
# Seleccionamos la primera hoja dinámicamente de los metadatos.
df_agent = pd.read_excel(DATA_PATH, sheet_name="Agent handled only volume")
df_hybrid = pd.read_excel(DATA_PATH, sheet_name="Hybrid Handled only volume")
df_bot = pd.read_excel(DATA_PATH, sheet_name="Bot only volume")

Lo anterior lo segmentamos por:

- modelo de gestión
- ruta de escalamiento
- nivel de automatización

Esto significa que Adidas realiza un seguimiento operativo de las conversaciones según el canal de resolución de problemas.


Lo que probablemente representa cada hoja

| Hoja | Significado |
| :--- | :--- |
| Agent handled only volume | Interacciones solo con humanos |
| Hybrid Handled only volume  | Bot + escalamiento humano |
| Bot only volume | Interacciones totalmente automatizadas |

## Por qué esto es extremadamente importante

Esta estructura nos permite analizar:

### A. Eficacia de la automatización
Podemos comparar:
- Lo que el bot resuelve **por sí solo**.
- Lo que requiere **escalamiento**.
- Lo que **evita** la automatización por completo.

### B. Complejidad de la intención/petición
Algunas intenciones/peticiones son:
- Fáciles de automatizar.
- Parcialmente automatizables.
- Imposibles de automatizar de forma segura.

Esta segmentación ayuda a identificarlas.

### C. Fugas de escalamiento
Los flujos híbridos son especialmente importantes porque suelen indicar:
1. Que el bot gestionó parcialmente la solicitud,
2. pero no la resolvió por completo.

> **Nota:** Este es uno de los mayores costes operativos en los sistemas de IA conversacional.

## Perspectiva/Insight estratégica

En última instancia, todo se reduce a:

> **“¿Cómo reducimos las escaladas híbridas innecesarias?”**

Ese es probablemente el verdadero objetivo empresarial.

**Un Insight muy importante.**

# Section 5 — Inspect Shapes

## Agent handled only volume

In [70]:
df_agent.shape

(184, 8)

In [87]:
df_agent.head()

,contact_reason,conteo_de_filas,percentage,unnamed_3,contact_reason1,sub_category,conteo_de_filas1,percentage1
0,Returns & Refunds,116308.0,0.292082,NaN,Customer Feedback,Left Blank,70731,0.177625
1,Existing Order,106339.0,0.267047,NaN,Support on Ordering,Explain how to order,52148,0.130958
2,Customer Feedback,71907.0,0.180579,NaN,Returns & Refunds,Left Blank,36494,0.091647
3,Support on Ordering,52426.0,0.131656,NaN,Returns & Refunds,Return status,30817,0.077390
4,Spam/No Contact,16204.0,0.040693,NaN,Existing Order,Size,20702,0.051989


In [72]:
df_agent.info()

<class 'pandas.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Contact Reason     16 non-null     str    
 1   Conteo de Filas    18 non-null     float64
 2   Percentage         18 non-null     float64
 3   Unnamed: 3         0 non-null      float64
 4   Contact Reason.1   184 non-null    str    
 5   Sub Category       184 non-null    str    
 6   Conteo de Filas.1  184 non-null    int64  
 7   Percentage.1       184 non-null    float64
dtypes: float64(4), int64(1), str(3)
memory usage: 11.6 KB


### Interpretación de la Estructura de Datos

El archivo analizado presenta las siguientes dimensiones estructurales:
* **184** registros (filas).
* **8** atributos (columnas).

**Premisa analítica fundamental:**
El conjunto de datos **NO** contiene microdatos o registros a nivel de sesión conversacional (unstructured/raw conversation logs). En su lugar, estamos ante un **dataset agregado de performance operacional**

Si tuviéramos acceso a datos transaccionales crudos a nivel de conversación, la estructura esperada reflejaría el siguiente esquema conceptual:

| conversation_id | timestamp  | customer_id | message              | intent       | resolution |
| --------------- | ---------- | ----------- | -------------------- | ------------ | ---------- |
| 92813           | 2026-01-01 | C182        | “Where is my order?” | order_status | resolved   |

Es decir, una arquitectura de datos granular caracterizada por:
- Una fila equivalente a una interacción o mensaje único.
- Timestamps de alta resolución temporal.
- Identificadores únicos (IDs de usuario, sesión e interacción).
- Metadatos conversacionales avanzados y textos libres (raw text strings).

Por el contrario, el dataset actual expone estructuras consolidadas como la siguiente:

| Contact Reason    | Conteo de Filas |
| ----------------- | --------------: |
| Returns & Refunds |          116308 |

La interpretación estadística correcta de este registro es: **“Se registraron 116,308 interacciones indexadas bajo la taxonomía de Returns & Refunds”**. No estamos auditando el flujo conversacional en vivo, sino su volumetría histórica etiquetada.

La presencia explícita de la métrica/campo `"Conteo de Filas"` confirma que el origen es un cubo de datos o reporte pre-agregado. Como consecuencia directa:
* **Ausencia de granularidad:** No existen identificadores (IDs) ni marcas de tiempo (timestamps).
* **Ausencia de texto libre:** No se dispone del registro literalizado de los chats ni de metadatos transaccionales crudos.
* **Información resumida:** El dataset provee métricas descriptivas consolidadas (Conteos, Porcentajes) y taxonomías categóricas ya normalizadas.

>  **Impacto Metodológico:** Esta restricción en la granularidad de los datos redefine por completo nuestra estrategia analítica y el alcance del proyecto.

---

### Redefinición del Enfoque Técnico

Dadas las características de la data disponible, se delimita estrictamente el alcance de la solución:

#### Lo que NO es viable abordar: 
* Entrenamiento, fine-tuning o reentrenamiento de modelos NLP.
* Generación de embeddings semánticos o clustering de texto no estructurado.
* Análisis de sentimiento o minería de transcripciones.
* Optimización directa de modelos de lenguaje natural (LLM).

#### Lo que SÍ se abordará:
* **Analítica Operacional de Contact Centers:** Diagnóstico de fallas en el funnel de atención.
* **Auditoría de KPIs Críticos:** Evaluación causal de las brechas en Contención, Repetitividad y Resolución (Containment, Repeat, Resolution).
* **Modelado de Oportunidades:** Matriz de priorización para automatización basada en el volumen de fuga hacia canales humanos.

#### Anatomía del Funnel de Atención Conversacional

```
[ Entrada Total de Contactos ]
              │
              ▼
   ┌─────────────────────┐
   │ 1. Clasificación    │  (¿El bot entiende el intent/motivo?)
   └──────────┬──────────┘
              │
              ▼
   ┌─────────────────────┐
   │ 2. Contención       │  (¿El bot retiene el caso sin derivar?)
   └──────────┬──────────┘
              │
              ▼
   ┌─────────────────────┐
   │ 3. Resolución       │  (¿El problema realmente se solucionó?)
   └─────────────────────┘

```

1. Capa de Entrada y Clasificación (Input & Intent Capture)
   - **Qué es:** El punto de partida donde el usuario interactúa y expresa su motivo de contacto (ej. "¿Dónde está mi pedido?").

   - **Métrica Clave:** Tasa de Clasificación de Intents. Evalúa el porcentaje de conversaciones donde el sistema de procesamiento de lenguaje natural (NLP) logra encasillar la duda en una categoría específica.

   - **Punto de Falla Típico:** Fugas por etiquetas como "Left Blank" o "Not defined by Bot". Si esta capa falla, el usuario cae directamente a las etapas inferiores de manera desordenada.

2. Capa de Contención (Deflection / Automation Layer)
   - **Qué es:** El filtro donde el chatbot procesa la solicitud utilizando flujos automatizados (informacionales o transaccionales) para evitar que la interacción requiera un costo operativo humano.

   - **Métrica Clave:** Containment Rate (Tasa de Contención). El porcentaje de contactos que completan su ciclo dentro del bot sin ser transferidos a un agente/asesor humano.

   - **Dinámica Operacional:** Una alta contención reduce la presión sobre el contact center, pero no es sinónimo de éxito si se fuerza al cliente a salir del canal sin una respuesta real.

3. Capa de Resolución (Fulfillment & Outcome)
   - **Qué es:** El fondo del embudo. Es la confirmación de que la necesidad del cliente fue satisfecha de manera efectiva en su primer contacto.

   - **Métrica Clave:** Resolution Rate (Tasa de Resolución) / First Contact Resolution (FCR).

   - **El Enlace Causal:** Si la tasa de resolución es baja, se genera una ruptura en el funnel. Esto provoca un rebote que infla artificialmente la Tasa de Repetitividad (Repeat Rate), obligando al usuario a volver a entrar al embudo y destruyendo la eficiencia del ecosistema.


## Hybrid Handled only volume

In [73]:
df_hybrid.shape

(208, 8)

In [74]:
df_hybrid.head()

,Contact Reason,Conteo de Filas,Percentage,Unnamed: 3,Contact Reason.1,Sub Category,Conteo de Filas.1,Percentage.1
0,Existing Order,113898.0,0.411416,NaN,Spam/No Contact,Left Blank,26688,0.096401
1,Returns & Refunds,74494.0,0.269083,NaN,Existing Order,Size,20571,0.074305
2,Spam/No Contact,26788.0,0.096762,NaN,Returns & Refunds,Return status,18098,0.065373
3,Payment,13149.0,0.047496,NaN,Existing Order,In transit,14978,0.054103
4,Product Information,11036.0,0.039864,NaN,Returns & Refunds,label Request,14028,0.050671


In [75]:
df_hybrid.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Contact Reason     19 non-null     str    
 1   Conteo de Filas    19 non-null     float64
 2   Percentage         19 non-null     float64
 3   Unnamed: 3         0 non-null      float64
 4   Contact Reason.1   208 non-null    str    
 5   Sub Category       208 non-null    str    
 6   Conteo de Filas.1  208 non-null    int64  
 7   Percentage.1       208 non-null    float64
dtypes: float64(4), int64(1), str(3)
memory usage: 13.1 KB


### Interpretación
Al analizar las matrices dimensionales de las hojas categóricas por canal, se observa una asimetría en el volumen de registros (*rows*):
* **Agent Only (Solo Humano):** Matrix de dimensión **(184, 8)** $\rightarrow$ 184 filas.
* **Hybrid (Bot + Humano):** Matrix de dimensión **(208, 8)** $\rightarrow$ 208 filas.

$$\text{Registros en Canal Híbrido (208)} > \text{Registros en Solo Agente (184)}$$

#### Formulación de Hipótesis Operacionales
Dado que este dataset se compone de datos agregados basados en taxonomías de contacto, una mayor cantidad de filas no implica mayor volumen de transacciones, sino una mayor cantidad de combinaciones de categorías exclusivas. A partir de esta asimetría estadística, se proponen dos hipótesis principales:

##### **Hipótesis 1:** Mayor Dispersión y Diversidad de la Intención (*Intent Diversity*)
Una tabla paramétrica con más registros indica una cobertura más amplia de escenarios de negocio. Esto sugiere que los usuarios que interactúan con el chatbot de forma inicial terminan cubriendo un espectro más diverso de problemáticas, subcategorías y combinaciones operativas que aquellos que se derivan o ingresan directamente con un agente humano.

##### **Hipótesis 2:** Alta Fragmentación Taxonómica en la Escalación
El canal híbrido muestra un fenómeno de **fragmentación taxonómica**. Mientras que el canal atendido puramente por agentes humanos opera bajo una clasificación más consolidada o simplificada, el flujo híbrido (donde el bot inicia y el humano finaliza) genera más subcategorías, *edge cases* o flujos de excepción. 

---

### Análisis de Complejidad: ¿Qué implica la "Fragmentación Taxonómica"?
La fragmentación taxonómica se traduce en una mayor granularidad y subdivisión del árbol de decisiones. 

* **Estructura Consolidada (Baja Complejidad):**
  `Returns` $\rightarrow$ `Orders` $\rightarrow$ `Payments`
* **Estructura Fragmentada / Canal Híbrido (Alta Complejidad):**
  - `Returns` $\rightarrow$ `Delayed Refund`
  - `Returns` $\rightarrow$ `Missing Label`
  - `Returns` $\rightarrow$ `Damaged Product`
  - `Returns` $\rightarrow$ `Refund Pending`

* Más granularidad.
* Más subdivisiones.
* Más complejidad operacional.



## Bot only volume

In [76]:
df_bot.shape

(77, 8)

In [77]:
df_bot.head()

,Contact Reason,Conteo de Filas,Percentage,Unnamed: 3,Contact Reason.1,Sub Category,Conteo de Filas.1,Percentage.1
0,Returns & Refunds,54282.0,0.439281,NaN,Returns & Refunds,How to return,19915,0.161164
1,Existing Order,50294.0,0.407008,NaN,Existing Order,Size,19235,0.155661
2,Payment,6618.0,0.053557,NaN,Returns & Refunds,Not defined by Bot,17245,0.139557
3,Not defined by Bot,4833.0,0.039111,NaN,Returns & Refunds,Return status,12934,0.104669
4,Apps & Website,2071.0,0.016760,NaN,Existing Order,Not defined by Bot,10052,0.081347


In [78]:
df_bot.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Contact Reason     15 non-null     str    
 1   Conteo de Filas    15 non-null     float64
 2   Percentage         15 non-null     float64
 3   Unnamed: 3         0 non-null      float64
 4   Contact Reason.1   77 non-null     str    
 5   Sub Category       77 non-null     str    
 6   Conteo de Filas.1  77 non-null     int64  
 7   Percentage.1       77 non-null     float64
dtypes: float64(4), int64(1), str(3)
memory usage: 4.9 KB


### Interpretación
**(77, 8)**

Muy pequeño.

**Esto sugiere firmemente que:**
El bot totalmente automatizado gestiona con éxito un **conjunto relativamente limitado de intenciones**.

> **Esto ya representa una importante información para el negocio.**

# Section 6 — Missing Values

In [80]:
df_agent.isna().sum()

Contact Reason       168
Conteo de Filas      166
Percentage           166
Unnamed: 3           184
Contact Reason.1       0
Sub Category           0
Conteo de Filas.1      0
Percentage.1           0
dtype: int64

In [79]:
df_hybrid.isna().sum()

Contact Reason       189
Conteo de Filas      189
Percentage           189
Unnamed: 3           208
Contact Reason.1       0
Sub Category           0
Conteo de Filas.1      0
Percentage.1           0
dtype: int64

In [81]:
df_bot.isna().sum()

Contact Reason       62
Conteo de Filas      62
Percentage           62
Unnamed: 3           77
Contact Reason.1      0
Sub Category          0
Conteo de Filas.1     0
Percentage.1          0
dtype: int64

## El descubrimiento más importante hasta ahora

> **El conjunto de datos está dividido estructuralmente en dos tablas dentro de cada hoja.**

### LADO IZQUIERDO
- Contact Reason
- Conteo de Filas
- Percentage

Esto es:
**HIGH-LEVEL CATEGORY AGGREGATION/ AGREGACIÓN DE CATEGORÍAS DE ALTO NIVEL** porque son categorías* MUY generales*, *no describen una acción específica* sino *dominios de negocio* y se *están sumando TODAS las conversaciones* pertenecientes a esa macro categoría.

Ejemplo:
- Returns & Refunds (Devoluciones y reembolsos): no es específico. Pueden existir `return status`, `refund pending`, `label request`, `damaged item`
- Existing Order (Orden existente): tampoco es específico. Puede incluir `tracking`, `delivery`, `in transit`, `modify address`
- Payment (Pago): 

Estas categorías funcionan como ***Categorías padre / macro categorías***

```
Returns & Refunds = return status + refund pending + label request + damaged item
```


### LADO DERECHO
- Contact Reason.1
- Sub Category
- Conteo de Filas.1
- Percentage.1

Esto es:
**SUBCATEGORY BREAKDOWN/DESGLOSE DE SUBCATEGORÍA***

Ejemplo:
- Return status
- Size
- Explain how to order

La propia estructura del Excel ya nos da la jerarquía.
| Contact Reason.1  | Sub Category  |
| ----------------- | ------------- |
| Returns & Refunds | Return status |
| Existing Order   | Size         |



### Por eso Hay Datos NULOS/NULLS

- Esto NO es **“datos erróneos/dirty data”**.
- Es un **error de formato de la hoja de cálculo**.
> Esto es sumamente importante.

**Ejemplo:**
| Contact Reason | Count/Conteo de Filas |
| :--- | :--- |
| Returns & Refunds | 116308 |

**A continuación, la hoja de cálculo contiene por separado:**
| Contact Reason.1 | Sub Category |
| :--- | :--- |
| Returns & Refunds | Return status |

**Por lo tanto, el archivo de Excel contiene:**
Dos tablas dinámicas independientes combinadas horizontalmente.

> **Esto explica la presencia de valores nulo en las columnas de la tabla de la izquierda..**
```
Contact Reason       168
Conteo de Filas      166

168 nulls
166 nulls
```

## La Columna "Unnamed: 3"
```
 3   Unnamed: 3         0 non-null      float64
```

### Interpretación

Casi con toda seguridad se trata de:
- una columna separadora visual en Excel
- Probablemente insertada para crear espacio entre las dos tablas dinámicas.

**Esta columna debe eliminarse inmediatamente.**

Esto es debido a que la tabla izquierda tiene menos filas reales (16 categorías.).

Ejemplo:

- Contact Reason
- Returns
- Orders
- Payment


Pero la tabla derecha tiene MUCHAS subcategorías. Entonces se rellena:

- las primeras filas con la tabla izquierda,
- y debajo quedan NaN.

**No es data faltante operacional. Es estructura visual del la hoja de Excel.**


### Hallazgos Operacionales Más Importantes

Ahora interpretemos los resultados reales del negocio.

#### A. Gestión exclusivamente por agentes/Agent-Only: Predominan las devoluciones y los pedidos existentes (Returns & Existing Orders)
-   **Returns & Refunds**
-   **Existing Order**

**Esto sugiere:**
La atención a transacciones complejas depende en gran medida de la *intervención humana*.

**Esto suele indicar:**
- Falta de integraciones con el sistema backend.
- Orquestación insuficiente del flujo de trabajo.
- Automatización deficiente de las transacciones.

> **Esto es estratégicamente importante.**

---

#### B. Gestión híbrida/Hybrid: También predominan los pedidos existentes/Existing Orders
**Muy importante.**

**Esto implica:**
1. El bot intenta ejecutar estos flujos de trabajo,
2. pero a menudo no puede completarlos.

> **Esto es una fuga de escalada conversacional.**

---

#### C. Intenciones más fuertes solo para Bot-Only
-   **How to return**
-   **Size**

Estas son:
- **Deterministas:** La respuesta sigue reglas fijas y repetibles.
- **Informativas:** Porque el usuario busca información, no busca ejecutar una transacción compleja. (“How do I return a product?” vs “My refund never arrived and my payment failed”)
- **De bajo riesgo:** si el bot responde incorrectamente, el impacto operacional suele ser menor. (“How to return” es manejable, pero un Error en pago/refund es más crítico; dinero, fraude, compliance, experiencia negativa severa.)
- **Repetibles:** porque miles de usuarios hacen exactamente la misma pregunta (“How to return”, “Return status”, “Size”), por ende son patrones altamente repetitivos e ideales para automatización.

> **Exactamente los tipos de intenciones que los LLM/chatbots manejan mejor.**

---

#### D. "No definido por el bot/ Not defined by Bot"
**Este es uno de los hallazgos más importantes en todo el conjunto de datos.**

**Ejemplo:**
- `Returns & Refunds` → **Not defined by Bot**

**Esto indica claramente:**
- Fallo en el reconocimiento de la intención/intent recognition failure.
- o limitaciones de taxonomía/taxonomy limitations.
- o brechas en el enrutamiento alternativo/fallback routing gaps.

> **Potencialmente las tres.**

## **Conclusión Section 6**:

- Ahora sabemos que los valores nulos, **no son** valores nulos operativos.
- Mas bien son **artifacts** del formato del panel de control de Excel.
- La estructura del Dataset:
    * Entendemos que cada hoja contiene:

**TABLE A — Category Summary**
| contact_reason | count | percentage |
| :--- | :--- | :--- |

<br>

**TABLE B — Subcategory Summary**
| contact_reason | sub_category | count | percentage |
| :--- | :--- | :--- | :--- |



*Dentro de la misma hoja de cálculo.*

---

### Esto implica cambios en nuestro siguiente paso

- No debemos analizar estas hojas directamente.
- **En su lugar:**
    * Debemos extraer y normalizar las tablas por separado.


---

### Lo más importante hasta ahora

> El conjunto de datos ya constituye una **capa de agregación de KPI operativos**, no datos de interacción sin procesar.

Es decir: estamos analizando:
- **Rendimiento del negocio/Business performance**
- **Cuellos de botella operativos/Operational bottlenecks**
- **Eficacia de la automatización/Automation effectiveness**

#### **NO:**
- Conversaciones individuales
- Incrustaciones de PLN
- Análisis de sentimientos

> **Esto define fundamentalmente el alcance de la solución.**

---

Nuestra recomendación final probablemente debería centrarse en:
| Area | Recommendation |
| :--- | :--- |
| **Intent classification** | Improve taxonomy + fallback handling |
| **Transactional flows** | Add Order Management System (OMS)/payment integrations |
| **Hybrid leakage** | Improve escalation decisioning |
| **FAQ automation** | Expand deterministic automation |
| **Analytics instrumentation** | Fix undefined intents |

# Section 7 — Standardize Column Names

In [82]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^\w]", "", regex=True)
    )
    return df

In [83]:
df_agent_clean = clean_columns(df_agent)

In [84]:
df_hybrid_clean = clean_columns(df_hybrid)

In [85]:
df_bot_clean = clean_columns(df_bot)

# Section 8 — Normalize Tables

Transformaremos las Tablas de Reporte de Excel en un Dataset analíticos limpio

Crearemos 4 datasets principales:

| Dataset                 | Nivel            |
| ----------------------- | ---------------- |
| `agent_category_df`     | categorías macro |
| `agent_subcategory_df`  | subcategorías    |
| `hybrid_category_df`    | categorías macro |
| `hybrid_subcategory_df` | subcategorías    |

Y luego consolidaremos todos

## Resultado final esperado

Terminaremos con datasets así:

### Category Dataset
| handling_channel | contact_reason    | volume | percentage |
| ---------------- | ----------------- | -----: | ---------: |
| agent_only       | Returns & Refunds | 116308 |      0.292 |


### Subcategory Dataset
| handling_channel | contact_reason | sub_category | volume | percentage |
| ---------------- | -------------- | ------------ | -----: | ---------: |
| bot_only         | Existing Order | Size         |  19235 |      0.156 |

**¿Por qué esto es MUY importante?**

Porque actualmente el Excel está optimizado para:
- visualización humana,
- reporting manual.

NO para:
- joins,
- dashboards,
- KPI analytics,
- Power BI,
- slicing/filtering,
- modeling.


La Section 8 tendrá esta estructura:

| Paso | Objetivo                         |
| ---- | -------------------------------- |
| 8.1  | Crear función para categorías    |
| 8.2  | Crear función para subcategorías |
| 8.3  | Normalizar Agent                 |
| 8.4  | Normalizar Hybrid                |
| 8.5  | Normalizar Bot                   |
| 8.6  | Consolidar datasets              |
| 8.7  | Validar integridad               |


## SECTION 8.1 — Create Category Extraction Function

In [91]:
def build_category_table(
    df: pd.DataFrame,
    handling_channel: str,
) -> pd.DataFrame:
    """
    Build normalized category-level dataset.

    Parameters
    ----------
    df : pd.DataFrame
        Raw worksheet dataframe.
    handling_channel : str
        Operational handling channel.

    Returns
    -------
    pd.DataFrame
        Clean category-level analytical dataset.
    """

    # Extrae SOLO la tabla izquierda porque esa es la **tabla macro categórica**.
    category_df = (
        df[
            [
                "contact_reason",
                "conteo_de_filas",
                "percentage",
            ]
        ]
        .dropna(subset=["contact_reason"]) # Elimina filas vacías
        .rename(
            columns={
                "conteo_de_filas": "volume", # Renombramos columnas: nombres más consistentes y limpios
            }
        )
        .copy()
    )

    # Agregamos handling_channel para comparar agent vs hybrid vs bot
    category_df["handling_channel"] = handling_channel

    category_df = category_df[
        [
            "handling_channel",
            "contact_reason",
            "volume",
            "percentage",
        ]
    ]

    return category_df

## SECTION 8.2 — Create Subcategory Function

In [92]:
def build_subcategory_table(
    df: pd.DataFrame,
    handling_channel: str,
) -> pd.DataFrame:
    """
    Build normalized subcategory-level dataset.

    Parameters
    ----------
    df : pd.DataFrame
        Raw worksheet dataframe.
    handling_channel : str
        Operational handling channel.

    Returns
    -------
    pd.DataFrame
        Clean subcategory-level analytical dataset.
    """

    # Aquí extraemos la tabla derecha.
    subcategory_df = (
        df[
            [
                "contact_reason1",
                "sub_category",
                "conteo_de_filas1",
                "percentage1",
            ]
        ]
        .rename(
            columns={
                "contact_reason1": "contact_reason",
                "conteo_de_filas1": "volume",
                "percentage1": "percentage",
            }
        )
        .copy()
    )

    subcategory_df["handling_channel"] = handling_channel

    subcategory_df = subcategory_df[
        [
            "handling_channel",
            "contact_reason",
            "sub_category",
            "volume",
            "percentage",
        ]
    ]

    return subcategory_df

## SECTION 8.3 — Build Agent Datasets

Construimos ambos datasets de Agent por categoría y subcategoría

**¿Por qué renombramos contact_reason1?**

Porque después de separar las tablas, ya NO necesitamos:
- .1
- columnas duplicadas.

Ahora **son datasets independientes**.

In [93]:
agent_category_df = build_category_table(
    df=df_agent_clean,
    handling_channel="agent_only",
)

agent_subcategory_df = build_subcategory_table(
    df=df_agent_clean,
    handling_channel="agent_only",
)

## SECTION 8.4 — Build Hybrid Datasets

In [94]:
hybrid_category_df = build_category_table(
    df=df_hybrid_clean,
    handling_channel="hybrid",
)

hybrid_subcategory_df = build_subcategory_table(
    df=df_hybrid_clean,
    handling_channel="hybrid",
)

## SECTION 8.5 — Build Bot Datasets

In [95]:
bot_category_df = build_category_table(
    df=df_bot_clean,
    handling_channel="bot_only",
)

bot_subcategory_df = build_subcategory_table(
    df=df_bot_clean,
    handling_channel="bot_only",
)

## SECTION 8.6 — Consolidate Datasets

Hacemos Merges de los datasets de categorías y subcategorías.

Con esto, ya tenemos un solo Dataset Analítico pues:

**Antes:**
- 3 hojas Excel separadas.

**Ahora:**
- 1 modelo analítico consistente.

### Category Dataset

In [96]:
categories_df = pd.concat(
    [
        agent_category_df,
        hybrid_category_df,
        bot_category_df,
    ],
    ignore_index=True,
)

### Subcategory Dataset

In [97]:
subcategories_df = pd.concat(
    [
        agent_subcategory_df,
        hybrid_subcategory_df,
        bot_subcategory_df,
    ],
    ignore_index=True,
)

## SECTION 8.7 — Validate Final Datasets

In [98]:
print(categories_df.shape)
print(subcategories_df.shape)

(50, 4)
(469, 5)


In [102]:
categories_df

,handling_channel,contact_reason,volume,percentage
0,agent_only,Returns & Refunds,116308.0,0.292082
1,agent_only,Existing Order,106339.0,0.267047
2,agent_only,Customer Feedback,71907.0,0.180579
3,agent_only,Support on Ordering,52426.0,0.131656
4,agent_only,Spam/No Contact,16204.0,0.040693
5,agent_only,Payment,12257.0,0.030781
6,agent_only,Membership,6712.0,0.016856
7,agent_only,Defective Returns Management,4494.0,0.011286
8,agent_only,Vouchers & Gift cards,3376.0,0.008478
9,agent_only,Product Information,2926.0,0.007348


In [101]:
subcategories_df

,handling_channel,contact_reason,sub_category,volume,percentage
0,agent_only,Customer Feedback,Left Blank,70731,0.177625
1,agent_only,Support on Ordering,Explain how to order,52148,0.130958
2,agent_only,Returns & Refunds,Left Blank,36494,0.091647
3,agent_only,Returns & Refunds,Return status,30817,0.077390
4,agent_only,Existing Order,Size,20702,0.051989
...,...,...,...,...,...
464,bot_only,Customer Feedback,Left Blank,1,0.000008
465,bot_only,Existing Order,Duplicate order,1,0.000008
466,bot_only,Membership,Cannot claim/ use reward,1,0.000008
467,bot_only,Membership,Voucher not working,1,0.000008
